# 00 — Environment Check
**Day 1, Step 1.** Confirms your VS Code + Jupyter setup can run everything the rest of the project needs before you touch any data.

Run every cell top to bottom. If a cell errors, fix it before moving on — don't skip ahead.

## 1. Confirm the kernel and Python version
In VS Code: open this notebook, click the kernel selector (top right), and choose the `.venv` you created for this project (see the setup steps in chat). Then run the cell below.

In [1]:
import sys
print("Python:", sys.version)
print("Executable:", sys.executable)
assert sys.version_info >= (3, 10), "Use Python 3.10+"


Python: 3.12.13 (main, Jun 23 2026, 15:23:43) [MSC v.1944 64 bit (AMD64)]
Executable: c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\.venv\Scripts\python.exe


## 2. Confirm required packages import cleanly
These are the libraries the whole 10-day plan depends on. If any import fails, run `pip install -r requirements.txt` in the terminal (with your venv active) and restart the kernel.

In [2]:
import importlib

required = [
    "pandas", "numpy", "scipy", "sklearn", "xgboost", "lightgbm",
    "matplotlib", "seaborn", "yaml", "pulp", "streamlit", "dotenv"
]

missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"OK    {pkg}")
    except ImportError:
        print(f"MISSING  {pkg}")
        missing.append(pkg)

if missing:
    print("\nInstall missing packages with: pip install -r requirements.txt")
else:
    print("\nAll required packages import cleanly.")


OK    pandas
OK    numpy
OK    scipy
OK    sklearn
OK    xgboost
OK    lightgbm
OK    matplotlib
OK    seaborn
OK    yaml
OK    pulp
OK    streamlit
OK    dotenv

All required packages import cleanly.


## 3. Confirm project paths resolve correctly
This checks that `src/utils/config_loader.py` can find `config/config.yaml` and that the paths inside it are correct, regardless of where the notebook is launched from.

In [3]:
import sys
from pathlib import Path

# Add project root to path so `from src... import ...` works from /notebooks
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config_loader import load_config, raw_path

cfg = load_config()
print("Project root:", PROJECT_ROOT)
print("Raw data dir:", cfg["paths"]["raw_dir"])
print("Configured raw files:", len(cfg["raw_files"]))


Project root: c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform
Raw data dir: data/raw/Datasets
Configured raw files: 12


RAW DATA CHECKING

In [4]:
from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nFiles/folders:")
for p in Path.cwd().iterdir():
    print(p)

Current directory:
c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\notebooks

Files/folders:
c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\notebooks\00_environment_check.ipynb
c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\notebooks\01_data_inventory_and_exploration.ipynb
c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\notebooks\02_data_cleaning_and_validation.ipynb


In [5]:
from pathlib import Path

raw_dir = PROJECT_ROOT / cfg["paths"]["raw_dir"]

print("Raw directory:", raw_dir.resolve())
print("Exists:", raw_dir.exists())

if raw_dir.exists():
    print("\nFiles:")
    for f in raw_dir.iterdir():
        print(f.name)

Raw directory: C:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\data\raw\Datasets
Exists: True

Files:
Dataset-10 GIS Terrain Dataset.csv
Dataset-11 (A) Actual Wind Farm Layout.csv
Dataset-11 (B) AI Candidate Layouts 10000.csv
Dataset-11 (C) Layout Position Features 10000.csv
Dataset-12 weather forecast 10000 rows.csv
Dataset-4 Long term wind climate dataset 20 years 10min.csv
Dataset-5 Turbine power curve dataset.csv
Dataset-6  Wind farm scada 1year.csv
Dataset-7 Turbine alarm event dataset 3year 50turbines 150k events.csv
Dataset-8  Wind turbine maintenance dataset 3year 50turbines 25000 records.csv
Dataset-9 Grid and Curtailment dataset 10000 rows.csv
wind_energy_dataset_100- 10000 Row.csv


## 4. Confirm every raw dataset file is actually present
This is the single most important Day-1 sanity check: a missing or misnamed file here will silently break every later notebook.

In [8]:
missing_files = []
for key, filename in cfg["raw_files"].items():
    p = raw_path(key, cfg)
    status = "OK" if p.exists() else "MISSING"
    size_mb = f"{p.stat().st_size / 1e6:,.1f} MB" if p.exists() else "-"
    print(f"{status:8s} {key:28s} {size_mb:>10s}   {filename}")
    if not p.exists():
        missing_files.append(filename)

assert not missing_files, f"Missing files: {missing_files}. Copy the datasets from Datasets.zip into data/raw/ first."
print("\nAll 12 raw datasets found.")


OK       long_term_climate              111.2 MB   Dataset-4 Long term wind climate dataset 20 years 10min.csv
OK       power_curve                      0.0 MB   Dataset-5 Turbine power curve dataset.csv
OK       scada                           12.0 MB   Dataset-6  Wind farm scada 1year.csv
OK       alarms                          39.7 MB   Dataset-7 Turbine alarm event dataset 3year 50turbines 150k events.csv
OK       maintenance                      4.9 MB   Dataset-8  Wind turbine maintenance dataset 3year 50turbines 25000 records.csv
OK       grid_curtailment                 1.3 MB   Dataset-9 Grid and Curtailment dataset 10000 rows.csv
OK       gis_terrain                      0.0 MB   Dataset-10 GIS Terrain Dataset.csv
OK       layout_actual                    0.0 MB   Dataset-11 (A) Actual Wind Farm Layout.csv
OK       layout_candidates                1.3 MB   Dataset-11 (B) AI Candidate Layouts 10000.csv
OK       layout_position_features         1.0 MB   Dataset-11 (C) Layout P

## 5. Confirm the Anthropic API key is set (needed later, Day 5)
Not required today, but worth checking early so you're not blocked on Day 5. Copy `.env.example` to `.env` and fill in your key when you have it — this step will just warn, not fail, if it's missing.

In [7]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")
key = os.getenv("ANTHROPIC_API_KEY")
if key:
    print("ANTHROPIC_API_KEY is set (", key[:8], "...)")
else:
    print("ANTHROPIC_API_KEY not set yet — fine for Day 1, needed by Day 5.")


ANTHROPIC_API_KEY not set yet — fine for Day 1, needed by Day 5.


---
**Day 1 checkpoint:** if every cell above ran without error, your environment is ready. Move to `01_data_inventory_and_exploration.ipynb`.